In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Uso de carga de trabajo

<span id="usage"></span>
El uso es una medida del tiempo que la QPU permanece bloqueada para tu carga de trabajo, y se calcula de forma diferente según el modo de ejecución que estés utilizando.

* El uso de sesión es el tiempo de reloj en que la sesión está activa. Consulta [Duración de la sesión](/guides/run-jobs-session#session-length) para más información sobre las transiciones de estado de la sesión.
* El uso por lote es la suma del tiempo cuántico (tiempo que el complejo QPU dedica a procesar tu trabajo) de todos los trabajos del lote.
* El uso de un trabajo individual es el tiempo cuántico que el trabajo emplea en su procesamiento.

Ten en cuenta que los trabajos fallidos o cancelados cuentan para tu uso en determinadas circunstancias; consulta la sección [Trabajos fallidos y cancelados](#failed-job) para más detalles.

Para los usuarios del plan de pago, el uso determina el costo de la carga de trabajo. Consulta [Administrar costos](/guides/manage-cost) para más detalles.

<span id="failed-job"></span>
## Uso para trabajos fallidos y cancelados
Cuando un trabajo falla o se cancela, el uso reportado es el siguiente:

* Modo trabajo o lote: El uso reportado es el tiempo que la QPU estuvo bloqueada para ejecutar tu carga de trabajo hasta el momento en que falló o fue cancelada. Por lo tanto, si el fallo o la cancelación ocurrió antes de que se estableciera el bloqueo, el uso reportado es cero. De lo contrario, el uso reportado de la carga de trabajo es la cantidad de uso acumulado antes de que fallara o se cancelara. Así, algunos trabajos fallidos no aparecen en tu uso reportado y otros sí.

* Modo sesión: El uso reportado es el tiempo de reloj que la sesión estuvo activa, independientemente del número de trabajos que fallen o se cancelen.

<span id="view-usage"></span>
## Consultar el uso real de una carga de trabajo
Una vez que una carga de trabajo ha finalizado, hay varias formas de ver su uso real:

- Ejecuta [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) o [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) en `qiskit-ibm-runtime` 0.30 o posterior. Si usas una versión anterior de `qiskit-ibm-runtime` (>= 0.23 y < 0.30), el uso aún puede encontrarse en `session.details()["usage_time"]` y `batch.details()["usage_time"]`.
- Usa [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) para ver el uso de un lote o sesión específicos.
- Usa [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) para ver el uso de un trabajo individual.

<span id="instance-usage"></span>
## Ver el uso de la instancia
Puedes ver el uso de una instancia en la página [Instancias](https://quantum.cloud.ibm.com/instances) o, para quienes tengan la autoridad correspondiente, en la página [Análisis](https://quantum.cloud.ibm.com/analytics). Ten en cuenta que las páginas pueden mostrar cifras de uso diferentes porque calculan el uso de forma distinta.

La página de Instancias muestra el uso en tiempo real de los últimos 28 días (período móvil), hasta el momento actual del día en curso. El uso de la página de Análisis se recalcula cada hora e incluye los últimos 28 días completos; es decir, muestra el uso desde las 00:00 de hace 28 días hasta hoy, al inicio de la hora en punto.

## Estimar el uso antes de enviar un trabajo
Aunque obtener una estimación local precisa es complicado por las operaciones adicionales realizadas para la supresión y mitigación de errores, puedes usar esta fórmula base para obtener una aproximación del uso estimado:

`<overhead por sub-trabajo> + (rep_delay + <longitud del circuito>) * <número de ejecuciones>`

- `<overhead por sub-trabajo>` es un overhead de aproximadamente 2 segundos por sub-trabajo. Esto incluye operaciones como cargar el payload en los controles electrónicos. Tu trabajo primitivo puede dividirse en varios sub-trabajos si es demasiado grande para que el motor de ejecución lo procese de una sola vez.
- `rep_delay` es una opción [personalizable por el usuario](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), y el valor predeterminado está dado por `backend.default_rep_delay`, que es 250 microsegundos en la mayoría de los backends de IBM Quantum. Ten en cuenta que reducir `rep_delay` disminuye el tiempo total de ejecución en la QPU, pero a expensas de una mayor tasa de error en la preparación de estados; consulta la guía [Ejecución con tasa de repetición dinámica](/guides/repetition-rate-execution) para más información.
- `<longitud del circuito>` es la longitud total de instrucciones. Cada instrucción toma un tiempo diferente en la QPU, por lo que la longitud total varía de un circuito a otro. Una medición, por ejemplo, puede tardar 56 veces más que una compuerta `x`. `backend.target[<instrucción>][<qubit>].duration` puede usarse para encontrar la duración exacta de cada instrucción. Una longitud de circuito típica suele estar entre 50 y 100 microsegundos. Si usas técnicas de supresión o mitigación de errores con las primitivas, se pueden insertar instrucciones adicionales en tu circuito, lo que aumentaría la longitud total del circuito.
    > **Note:** La [opción experimental `scheduler_timing`](/guides/visualize-circuit-timing) devuelve el tiempo total del circuito, pero este NO es el tiempo utilizado para la facturación.
- `<número de ejecuciones>` es el número total de circuitos multiplicado por el número de shots, donde los circuitos son los generados después de que los elementos PUB se propagan.
    - Si usas técnicas de mitigación de errores con las primitivas, es posible que se ejecuten circuitos adicionales como parte del proceso de mitigación, lo que aumentaría el número total de ejecuciones. Además, las técnicas avanzadas de mitigación de errores como PEA y PEC tienen un overhead mucho mayor porque requieren ejecutar circuitos para el aprendizaje del ruido.
    - Estimator agrupa observables que conmutan qubit a qubit, lo que reduce el número de ejecuciones.

Si no usas ninguna técnica avanzada de mitigación de errores ni un `rep_delay` personalizado, puedes usar `2+0.00035*<número de ejecuciones>` como fórmula rápida.

## Próximos pasos
> **Tip:** - Revisa estos consejos: [Minimizar el tiempo de ejecución de trabajos](minimize-time).
>     - Establece el [Tiempo máximo de ejecución](max-execution-time).
>     - Aprende a transpilar localmente en la sección [Transpile](./transpile/).
>     - Prueba la guía [Comparar configuraciones del Transpiler](/guides/circuit-transpilation-settings).